# Каталог параметров оценки стоимости ML/AI-решений

Ноутбук последовательно читает кейсы из `data/train`, формирует по первому кейсу начальный каталог параметров, а затем дополняет и уточняет его на следующих кейсах. Результаты сохраняются в `output/cost_estimation_parameters.csv`, журнал обработки — в `output/processing_log.csv`.

Перед запуском создайте `.env` с `OPENAI_API_KEY=...`. При необходимости модель можно задать переменной `OPENAI_MODEL`.

In [1]:
# При необходимости раскомментируйте:
# %pip install openai python-dotenv pandas

from pathlib import Path
from datetime import datetime, timezone
from numbers import Integral
import json
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

## Настройки

`LOG_REUSE_EVENTS` управляет записью событий `reuse` в журнал. На обновление списка кейсов в итоговой таблице эта настройка не влияет.

In [9]:
TRAIN_DIR = Path("data/train")
OUTPUT_DIR = Path("output")
PARAMETERS_CSV = OUTPUT_DIR / "cost_estimation_parameters.csv"
LOG_CSV = OUTPUT_DIR / "processing_log.csv"

LOG_REUSE_EVENTS = False
load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
MAX_API_ATTEMPTS = 3
MIN_FIRST_CASE_PARAMETERS = 15
MAX_FIRST_CASE_PARAMETERS = 80

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("В .env не найден OPENAI_API_KEY")

client = OpenAI()
case_paths = sorted(path for path in TRAIN_DIR.iterdir() if path.is_file())
if not case_paths:
    raise RuntimeError(f"В {TRAIN_DIR} не найдено ни одного кейса")

print(f"Модель: {MODEL}")
print(f"Найдено кейсов: {len(case_paths)}")
print(f"Логировать reuse: {LOG_REUSE_EVENTS}")

Модель: gpt-5-mini
Найдено кейсов: 3
Логировать reuse: False


## Схема данных и промпты

In [3]:
PARAMETER_COLUMNS = [
    "параметр",
    "подробное описание параметра",
    "примеры значений",
    "важность (1-10)",
    "кейсы в которых он использовался",
    "когда применять этот параметр оценки",
    "когда не применять этот параметр",
]

LOG_COLUMNS = [
    "timestamp_utc", "case_id", "case_file", "action",
    "parameter", "changed_fields", "before", "after",
    "reason", "model", "prompt_version", "status",
]

FIELD_MAP = {
    "parameter": "параметр",
    "description": "подробное описание параметра",
    "example_values": "примеры значений",
    "importance": "важность (1-10)",
    "cases": "кейсы в которых он использовался",
    "apply_when": "когда применять этот параметр оценки",
    "do_not_apply_when": "когда не применять этот параметр",
}

PROMPT_VERSION = "1.0"

In [4]:
FIRST_SYSTEM_PROMPT = """
Ты — ведущий системный аналитик и эксперт по оценке стоимости разработки ML-, LLM-, data- и backend-решений.
По описанию реализованного кейса сформируй каталог параметров, которые нужно уточнить для достаточно точной оценки стоимости похожего решения.

Правила:
1. Пиши только на русском языке.
2. Не оценивай стоимость в деньгах или человеко-днях.
3. Каждый параметр — один атомарный, независимо уточняемый фактор стоимости.
4. Покрой аналитику, данные, модели/LLM/RAG, разработку, интеграции, инфраструктуру, MLOps, тестирование, безопасность, запуск и сопровождение.
5. Не создавай смысловые дубли. Названия делай короткими и пригодными для других кейсов.
6. Описание объясняет, что уточнять и почему это влияет на стоимость.
7. Примеры содержат реалистичные категории или диапазоны для разных проектов, разделенные точкой с запятой.
8. Важность — целое число 1–10: 1–3 слабое, 4–6 заметное, 7–8 сильное, 9–10 ключевое влияние.
9. Явно укажи условия применения и неприменения.
10. Сведения с пометками inferred, uncertain и NO INFO не считай фактами: формулируй их как то, что следует уточнить.
11. Сформируй от 35 до 60 содержательных параметров.
12. Верни только корректный JSON без Markdown.

Формат: {"parameters": [{"parameter": "...", "description": "...", "example_values": "...", "importance": 1, "cases": ["case_id"], "apply_when": "...", "do_not_apply_when": "...", "evidence": "...", "confidence": "high|medium|low"}]}
""".strip()

ITERATION_SYSTEM_PROMPT = """
Ты — ведущий системный аналитик и эксперт по оценке стоимости разработки ML-, LLM-, data- и backend-решений.
Ты поддерживаешь единый каталог атомарных параметров. Для факторов нового кейса выбери: reuse — параметр полностью подходит; update — его нужно расширить; add — подходящего параметра нет.

Правила:
1. Пиши только на русском языке и верни только корректный JSON без Markdown.
2. Не создавай новый параметр, если существующий отличается лишь формулировкой.
3. Не объединяй независимые факторы, не удаляй параметры и не переименовывай их.
4. При update сохраняй полезную старую информацию и расширяй ее.
5. Важность — целое число 1–10 и отражает общее влияние на стоимость. Изменение важности обязательно объясни.
6. При reuse и update используй точное существующее название target_parameter.
7. Каждый применимый существующий параметр должен получить reuse или update; каждый отсутствующий — add.
8. Сведения inferred, uncertain и NO INFO не считай установленными фактами.

Формат: {"case_summary": "...", "operations": [{"action": "reuse|update|add", "target_parameter": "...", "reason": "...", "evidence": "...", "changed_fields": [], "parameter_data": null}]}
Для update/add в parameter_data передай полный параметр с полями: parameter, description, example_values, importance, cases, apply_when, do_not_apply_when. Для reuse parameter_data должен быть null.
""".strip()

## Вызов OpenAI и проверки

Ответ запрашивается в JSON-формате и дополнительно проверяется в Python. При ошибке формата запрос повторяется с текстом ошибки.

In [5]:
def call_openai_json(system_prompt, user_prompt):
    last_error = None
    for attempt in range(1, MAX_API_ATTEMPTS + 1):
        correction = "" if last_error is None else f"\n\nПредыдущий ответ не прошел проверку: {last_error}. Исправь формат."
        response = client.responses.create(
            model=MODEL,
            instructions=system_prompt,
            input=user_prompt + correction,
            text={"format": {"type": "json_object"}},
        )
        try:
            return json.loads(response.output_text)
        except json.JSONDecodeError as error:
            last_error = str(error)
            print(f"Попытка {attempt}: модель вернула некорректный JSON")
    raise RuntimeError(f"Не удалось получить корректный JSON: {last_error}")


def normalize_parameter(raw, case_id):
    missing = [key for key in FIELD_MAP if key not in raw]
    if missing:
        raise ValueError(f"У параметра отсутствуют поля: {missing}")

    importance = int(raw["importance"])
    if not 1 <= importance <= 10:
        raise ValueError(f"Важность вне диапазона 1–10: {importance}")

    cases = raw["cases"] if isinstance(raw["cases"], list) else str(raw["cases"]).split(";")
    cases = sorted({str(value).strip() for value in cases if str(value).strip()} | {case_id})
    row = {
        "параметр": str(raw["parameter"]).strip(),
        "подробное описание параметра": str(raw["description"]).strip(),
        "примеры значений": str(raw["example_values"]).strip(),
        "важность (1-10)": importance,
        "кейсы в которых он использовался": "; ".join(cases),
        "когда применять этот параметр оценки": str(raw["apply_when"]).strip(),
        "когда не применять этот параметр": str(raw["do_not_apply_when"]).strip(),
    }
    empty = [column for column, value in row.items() if value == ""]
    if empty:
        raise ValueError(f"У параметра пустые поля: {empty}")
    return row


def validate_catalog(frame, known_case_ids):
    if list(frame.columns) != PARAMETER_COLUMNS:
        raise ValueError("Нарушен состав или порядок столбцов каталога")
    if frame.empty:
        raise ValueError("Каталог пуст")
    if frame["параметр"].duplicated().any():
        duplicates = frame.loc[frame["параметр"].duplicated(), "параметр"].tolist()
        raise ValueError(f"Повторяющиеся параметры: {duplicates}")
    if not frame["важность (1-10)"].map(lambda value: isinstance(value, Integral) and 1 <= value <= 10).all():
        raise ValueError("Важность должна быть целым числом от 1 до 10")
    for value in frame["кейсы в которых он использовался"]:
        cases = {item.strip() for item in value.split(";") if item.strip()}
        if not cases or not cases <= known_case_ids:
            raise ValueError(f"Некорректный список кейсов: {value}")


def to_api_parameter(row):
    return {
        "parameter": row["параметр"],
        "description": row["подробное описание параметра"],
        "example_values": row["примеры значений"],
        "importance": int(row["важность (1-10)"]),
        "cases": [value.strip() for value in row["кейсы в которых он использовался"].split(";")],
        "apply_when": row["когда применять этот параметр оценки"],
        "do_not_apply_when": row["когда не применять этот параметр"],
    }

In [6]:
processing_log = []


def add_log(case_id, case_file, action, parameter="", changed_fields=None, before=None, after=None, reason="", status="success"):
    if action == "reuse" and not LOG_REUSE_EVENTS:
        return
    processing_log.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "case_id": case_id,
        "case_file": str(case_file),
        "action": action,
        "parameter": parameter,
        "changed_fields": "; ".join(changed_fields or []),
        "before": json.dumps(before, ensure_ascii=False) if before is not None else "",
        "after": json.dumps(after, ensure_ascii=False) if after is not None else "",
        "reason": reason,
        "model": MODEL,
        "prompt_version": PROMPT_VERSION,
        "status": status,
    })


def save_results(parameters_df):
    parameters_df.to_csv(PARAMETERS_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame(processing_log, columns=LOG_COLUMNS).to_csv(LOG_CSV, index=False, encoding="utf-8-sig")

## Первый кейс: создание каталога

In [10]:
known_case_ids = {path.stem for path in case_paths}
first_path = case_paths[0]
first_case_id = first_path.stem
first_text = first_path.read_text(encoding="utf-8")

add_log(first_case_id, first_path, "case_started", reason="Создание первоначального каталога")
first_user_prompt = f"""
Идентификатор кейса: {first_case_id}
Сформируй первоначальный каталог параметров оценки стоимости похожего решения.
Текст кейса:
<case>
{first_text}
</case>
Верни результат строго в формате JSON.

""".strip()

first_result = call_openai_json(FIRST_SYSTEM_PROMPT, first_user_prompt)
raw_parameters = first_result.get("parameters", [])
if not MIN_FIRST_CASE_PARAMETERS <= len(raw_parameters) <= MAX_FIRST_CASE_PARAMETERS:
    raise ValueError(f"Первый ответ содержит {len(raw_parameters)} параметров, ожидалось {MIN_FIRST_CASE_PARAMETERS}–{MAX_FIRST_CASE_PARAMETERS}")

rows = [normalize_parameter(item, first_case_id) for item in raw_parameters]
parameters_df = pd.DataFrame(rows, columns=PARAMETER_COLUMNS)
validate_catalog(parameters_df, known_case_ids)
for row in rows:
    add_log(first_case_id, first_path, "add", parameter=row["параметр"], after=row, reason="Параметр выделен из первого кейса")
add_log(first_case_id, first_path, "case_completed", reason=f"Создано параметров: {len(parameters_df)}")
save_results(parameters_df)
print(f"{first_case_id}: создано {len(parameters_df)} параметров")

case_000_disdoc: создано 71 параметров


## Следующие кейсы: reuse, update и add

После каждого кейса CSV перезаписываются, поэтому уже обработанные результаты сохранятся даже при ошибке на следующем API-вызове.

In [ ]:
for case_path in case_paths[1:]:
    case_id = case_path.stem
    case_text = case_path.read_text(encoding="utf-8")
    catalog_json = json.dumps(
        [to_api_parameter(row) for _, row in parameters_df.iterrows()],
        ensure_ascii=False,
    )
    user_prompt = f"""
Идентификатор нового кейса: {case_id}
Определи применимые параметры и необходимые изменения каталога.

<catalog>
{catalog_json}
</catalog>

<case>
{case_text}
</case>
Верни результат строго в формате JSON.

""".strip()

    add_log(case_id, case_path, "case_started", reason="Итеративное обновление каталога")
    result = call_openai_json(ITERATION_SYSTEM_PROMPT, user_prompt)
    operations = result.get("operations", [])
    if not operations:
        raise ValueError(f"Для {case_id} модель не вернула операций")

    for operation in operations:
        action = operation.get("action")
        target = str(operation.get("target_parameter", "")).strip()
        reason = str(operation.get("reason", "")).strip()
        matches = parameters_df.index[parameters_df["параметр"] == target].tolist()

        if action == "reuse":
            if len(matches) != 1:
                raise ValueError(f"reuse с неизвестным параметром: {target}")
            index = matches[0]
            before = parameters_df.loc[index].to_dict()
            cases = {value.strip() for value in before["кейсы в которых он использовался"].split(";") if value.strip()}
            cases.add(case_id)
            parameters_df.at[index, "кейсы в которых он использовался"] = "; ".join(sorted(cases))
            after = parameters_df.loc[index].to_dict()
            add_log(case_id, case_path, "reuse", target, ["кейсы в которых он использовался"], before, after, reason)

        elif action == "update":
            if len(matches) != 1:
                raise ValueError(f"update с неизвестным параметром: {target}")
            index = matches[0]
            before = parameters_df.loc[index].to_dict()
            after = normalize_parameter(operation.get("parameter_data") or {}, case_id)
            if after["параметр"] != target:
                raise ValueError(f"При update нельзя переименовывать параметр {target}")
            old_cases = {value.strip() for value in before["кейсы в которых он использовался"].split(";") if value.strip()}
            new_cases = {value.strip() for value in after["кейсы в которых он использовался"].split(";") if value.strip()}
            after["кейсы в которых он использовался"] = "; ".join(sorted(old_cases | new_cases | {case_id}))
            parameters_df.loc[index] = after
            changed = [column for column in PARAMETER_COLUMNS if before[column] != after[column]]
            add_log(case_id, case_path, "update", target, changed, before, after, reason)

        elif action == "add":
            if matches:
                raise ValueError(f"Нельзя повторно добавить параметр: {target}")
            after = normalize_parameter(operation.get("parameter_data") or {}, case_id)
            if after["параметр"] != target:
                raise ValueError(f"target_parameter не совпадает с новым параметром: {target}")
            parameters_df.loc[len(parameters_df)] = after
            add_log(case_id, case_path, "add", target, PARAMETER_COLUMNS, None, after, reason)

        else:
            raise ValueError(f"Неизвестное действие: {action}")

    parameters_df = parameters_df.sort_values("параметр").reset_index(drop=True)
    validate_catalog(parameters_df, known_case_ids)
    add_log(case_id, case_path, "case_completed", reason=f"Применено операций: {len(operations)}")
    save_results(parameters_df)
    print(f"{case_id}: операций {len(operations)}, всего параметров {len(parameters_df)}")

## Итоговая проверка и просмотр результатов

In [8]:
validate_catalog(parameters_df, known_case_ids)
save_results(parameters_df)

reloaded_parameters = pd.read_csv(PARAMETERS_CSV, encoding="utf-8-sig")
reloaded_log = pd.read_csv(LOG_CSV, encoding="utf-8-sig")
assert list(reloaded_parameters.columns) == PARAMETER_COLUMNS
assert list(reloaded_log.columns) == LOG_COLUMNS

print(f"Обработано кейсов: {len(case_paths)}")
print(f"Итоговых параметров: {len(parameters_df)}")
print(f"Записей в логе: {len(processing_log)}")
print(f"Каталог: {PARAMETERS_CSV}")
print(f"Лог: {LOG_CSV}")
display(parameters_df.head(10))
display(pd.DataFrame(processing_log, columns=LOG_COLUMNS).tail(10))

NameError: name 'parameters_df' is not defined